# 🧪 W2-D7 概念实验：大模型（推理）全景 —— 亲手搭一个 mini-GPT 前向

> 配套阅读：`第2周-Day7-第二周总复习.md`（知识图谱与流程总结在那边）
> 复习日不抄讲义：把 W1-W2 学的 **Tokenizer → Embedding → 位置编码 → 因果注意力 →
> FFN/LayerNorm/残差 → logits → 采样** 串成一条真的能跑的前向链路
>
> 实验环境：纯 numpy + matplotlib，模型只有几十 KB——但结构与 GPT 同构。

## 第 1 环：Tokenizer —— 文本 ↔ token id

真实大模型用 BPE/WordPiece（几万词表）。这里用字符级做原理演示：
**同样的信息，词表越小序列越长**——这正是 BPE 存在的理由。

In [ ]:
import numpy as np

corpus = "猫追老鼠。狗追皮球。它追它们。猫累了，狗也累了。追了一下午。"
vocab = sorted(set(corpus))
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}

def encode(text): return [stoi[c] for c in text]
def decode(ids):  return "".join(itos[i] for i in ids)

ids = encode("猫追老鼠。")
print(f"词表大小: {len(vocab)}（GPT-4 用 ~10万 BPE，LLaMA-3 用 12.8万）")
print(f"编码 '猫追老鼠。' → {ids}")
print(f"解码回文: {decode(ids)}  ✓ 往返一致")
print("→ 字符级词表仅 17 个，一句话要 5 个 token；BPE 合并常见片段，同样意思用更少 token")

## 第 2 环：Embedding + 位置编码 + 因果注意力

查表得词向量 → 加正弦位置编码 → 过带因果掩码的单头注意力。
注意 shape 全程不变：这是"能堆 96 层"的前提。

In [ ]:
rng = np.random.default_rng(2024)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def sinusoidal_pe(pos, d):
    pe = np.zeros(d)
    for i in range(0, d, 2):
        freq = 1.0 / (10000 ** (i / d))
        pe[i], pe[i+1] = np.sin(pos * freq), np.cos(pos * freq)
    return pe

d = 32
E = rng.normal(scale=0.5, size=(len(vocab), d))
text = "猫追老鼠。"
ids = encode(text)
X = E[ids]                                    # (5, 32)
H = X + np.array([sinusoidal_pe(p, d) for p in range(len(ids))])

Wq = rng.normal(scale=0.5, size=(d, d)); Wk = rng.normal(scale=0.5, size=(d, d))
Wv = rng.normal(scale=0.5, size=(d, d))
Q, K, V = H @ Wq, H @ Wk, H @ Wv
causal = np.tril(np.ones((len(ids), len(ids)), dtype=int))
W_att = softmax(np.where(causal, Q @ K.T / np.sqrt(d), -1e9), axis=1)
attn_out = W_att @ V

for name, arr in [("ids", np.array(ids)), ("Embedding X", X), ("+PE H", H),
                  ("attn out", attn_out)]:
    print(f"{name:<12} shape {arr.shape}")
print("\n'。'(最后一词) 的注意力:", dict(zip(list(text), W_att[-1].round(3))), " ← 只分给左边")

## 第 3 环：FFN + LayerNorm + 残差 —— 层内的"后处理"

注意力负责"混词"，FFN 负责"深加工"（4 倍扩张 + GELU），
LayerNorm 把数值拉回稳定区间，残差给梯度修高速公路。

In [ ]:
def layer_norm(x, gamma=1.0, beta=0.0, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    sigma = x.std(axis=-1, keepdims=True)
    return gamma * (x - mu) / (sigma + eps) + beta

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

d_ff = 4 * d
W1 = rng.normal(scale=0.4, size=(d, d_ff))
W2 = rng.normal(scale=0.4, size=(d_ff, d))

h1 = layer_norm(H + attn_out)                  # 残差1 + LN（Post-LN 写法）
ffn = gelu(h1 @ W1) @ W2                       # 32 → 128 → 32
block_out = layer_norm(h1 + ffn)               # 残差2 + LN

print("LN 前 H 的行均值/标准差:", H.mean(axis=1)[:2].round(2), H.std(axis=1)[:2].round(3))
print("LN 后       :", h1.mean(axis=1)[:2].round(6), h1.std(axis=1)[:2].round(3), " ← 每行归一")
print(f"FFN: {d} → {d_ff} → {d}（先扩张 4 倍再压缩，参数占每层的 2/3）")
print("残差连接: block_out = LN(h1 + ffn)，梯度可以直通输入")
print("block 输出 shape:", block_out.shape, " ← 又回到 (n, d)，可以继续堆下一层")

## 第 4 环：logits → 采样 —— 温度与 top-k 如何塑造输出

输出投影复用 Embedding 矩阵（weight tying）。对最后一个位置的 logits 做
softmax 得到下一词分布：温度 T 压平/锐化分布，top-k 砍掉长尾。

In [ ]:
logits = block_out @ E.T                      # (n, V)：weight tying，省一份参数
last = logits[-1]

def top_k_probs(logits, T=1.0, k=None):
    p = softmax(logits / T)
    if k is not None:
        idx = np.argsort(p)[::-1][:k]          # 只保留 top-k
        mask = np.zeros_like(p); mask[idx] = p[idx]
        p = mask / mask.sum()
    return p

print("最后一个词是 '。'，模型预测下一个词（未训练，看机制）：\n")
for T in [0.1, 1.0, 5.0]:
    p = softmax(last / T)
    top = np.argsort(p)[::-1][:3]
    ent = -(p * np.log(p + 1e-12)).sum()
    print(f"T={T:<4} 熵={ent:5.2f}  top3: " +
          "  ".join(f"{itos[i]}({p[i]:.2f})" for i in top))
print("\n→ T→0 退化为 greedy（必然采样），T 大则更随机（更有多样性、更容易跑偏）")

p_topk = top_k_probs(last, T=1.0, k=3)
print("top-k=3 后只剩:", "  ".join(f"{itos[i]}({p_topk[i]:.2f})" for i in np.argsort(p_topk)[::-1][:3]))

## 第 5 环：全景收官 —— 分布可视化

把第 4 环的三个温度画在同一张图上：同一组 logits，采样策略不同，
输出的"性格"完全不同。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

cands = np.argsort(softmax(last))[::-1][:6]
cand_tokens = [itos[i] for i in cands]

fig, ax = plt.subplots(figsize=(8.5, 4.2))
x = np.arange(len(cands))
w = 0.26
for j, T in enumerate([0.1, 1.0, 5.0]):
    p = softmax(last / T)[cands]
    ax.bar(x + (j-1)*w, p, width=w, label=f"T = {T}")
ax.set_xticks(x, cand_tokens)
ax.set_xlabel("候选 token"); ax.set_ylabel("采样概率")
ax.set_title("同一 logits、不同温度：T=0.1 接近确定性，T=5 接近均匀")
ax.legend(); plt.tight_layout(); plt.show()

print("W1-W2 全景: Tokenizer → Embedding → +PE → 因果注意力 → FFN/LN/残差 → logits → 采样")
print("这条链路上每一环都有优化旋钮: 分词效率 / RoPE / KV-Cache+GQA / Flash-Attention / 采样参数")

## 结论

| 环节 | 本 notebook 实现 | 对应优化技术 |
|---|---|---|
| Tokenizer | 字符级查表 | BPE / SentencePiece（W2-D2） |
| 位置 | 正弦绝对编码 | RoPE / ALiBi（W2-D4） |
| 注意力 | 因果掩码单头 | KV Cache / GQA / Flash Attention（W2-D5） |
| FFN/LN/残差 | Post-LN + GELU | Pre-LN / SwiGLU |
| 采样 | 温度 + top-k | top-p / repetition penalty |

→ 深入阅读：同目录 `.md` 版本第二节（从数据到推理的完整流程图）